# 05 — Phoenix Sentinel-2 Composites

Validates the monthly cloud-masked median composites built by `ug composite generate`.

**Sections**
1. RGB + NDVI for 4 sample months
2. Monthly coverage % (valid pixel fraction)
3. Flag months with < 70 % valid coverage

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from dotenv import load_dotenv

load_dotenv()

from urbangrowth.config import data_path, get_pipeline

CITY = 'phoenix'
pipe = get_pipeline()
COMPOSITE_DIR = data_path(pipe['processed_data_subdirs']['composites'], CITY)

# Band index mapping (0-based) in the 9-band COG
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']
BAND_IDX = {b: i for i, b in enumerate(BANDS)}

print(f'Composite directory: {COMPOSITE_DIR}')
all_tifs = sorted(COMPOSITE_DIR.glob('*.tif'))
print(f'Total composites found: {len(all_tifs)}')

## 1. RGB + NDVI — 4 Sample Months

Shows true-colour RGB (B04/B03/B02) and NDVI for four evenly spaced months
across the archive.

In [ ]:
def load_composite(path: Path) -> xr.DataArray:
    """Load a composite COG, drop the band_description coord that rioxarray adds."""
    da = rioxarray.open_rasterio(path, chunks={'x': 2048, 'y': 2048})
    # rioxarray uses 1-based integer band indices by default; rename to our labels
    da = da.assign_coords(band=BANDS)
    return da


def stretch(arr: np.ndarray, lo: float = 2, hi: float = 98) -> np.ndarray:
    """Percentile linear stretch for display."""
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        return arr
    vmin, vmax = np.percentile(valid, [lo, hi])
    return np.clip((arr - vmin) / (vmax - vmin + 1e-9), 0, 1)


def composite_date(p: Path) -> str:
    return p.stem  # 'YYYY-MM'


# Pick 4 evenly spaced months
if len(all_tifs) == 0:
    print('No composites found — run: ug composite generate --city phoenix')
else:
    step = max(1, len(all_tifs) // 4)
    samples = all_tifs[::step][:4]

    fig, axes = plt.subplots(len(samples), 2, figsize=(14, 5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    for row, tif in enumerate(samples):
        da = load_composite(tif).compute()

        # RGB — B04 (red), B03 (green), B02 (blue)
        rgb = np.stack([
            stretch(da.sel(band='B04').values.astype(float)),
            stretch(da.sel(band='B03').values.astype(float)),
            stretch(da.sel(band='B02').values.astype(float)),
        ], axis=-1)

        axes[row, 0].imshow(rgb, interpolation='bilinear')
        axes[row, 0].set_title(f"{composite_date(tif)} — RGB", fontsize=11)
        axes[row, 0].axis('off')

        # NDVI
        ndvi = da.sel(band='NDVI').values.astype(float)
        im = axes[row, 1].imshow(
            ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.8, interpolation='bilinear'
        )
        plt.colorbar(im, ax=axes[row, 1], fraction=0.03, pad=0.04, label='NDVI')
        axes[row, 1].set_title(f"{composite_date(tif)} — NDVI", fontsize=11)
        axes[row, 1].axis('off')

    fig.suptitle(f'Phoenix Sentinel-2 Composites — Sample Months', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 2. Monthly Coverage %

For each composite, the coverage percentage is the fraction of pixels that
have a finite (non-NaN) value in the B02 band after cloud masking and median
aggregation.  Low coverage months have high cloud or snow frequency.

In [ ]:
COVERAGE_THRESHOLD = 70.0   # flag months below this

records = []
for tif in all_tifs:
    da = rioxarray.open_rasterio(tif)
    da = da.assign_coords(band=BANDS)
    b02 = da.sel(band='B02').values.astype(float)
    total = b02.size
    valid = int(np.isfinite(b02).sum())
    coverage = valid / total * 100 if total > 0 else 0.0
    records.append({
        'month': composite_date(tif),
        'coverage_pct': round(coverage, 1),
        'valid_px': valid,
        'total_px': total,
    })

cov_df = pd.DataFrame(records).set_index('month')

if cov_df.empty:
    print('No composites to evaluate.')
else:
    fig, ax = plt.subplots(figsize=(16, 4))
    colors = [
        '#d62728' if c < COVERAGE_THRESHOLD else '#2ca02c'
        for c in cov_df['coverage_pct']
    ]
    ax.bar(range(len(cov_df)), cov_df['coverage_pct'], color=colors, width=0.85)
    ax.axhline(COVERAGE_THRESHOLD, color='orange', linestyle='--', linewidth=1.5,
               label=f'{COVERAGE_THRESHOLD}% threshold')

    # x-axis labels — show every 6th month
    tick_positions = list(range(0, len(cov_df), 6))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([cov_df.index[i] for i in tick_positions], rotation=45, ha='right')

    ax.set_ylabel('Coverage %')
    ax.set_ylim(0, 105)
    ax.set_title(f'Phoenix — monthly valid-pixel coverage (green ≥ {COVERAGE_THRESHOLD}%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'\nMonths with coverage < {COVERAGE_THRESHOLD}%: '
          f'{(cov_df.coverage_pct < COVERAGE_THRESHOLD).sum()} / {len(cov_df)}')

## 3. Low-Coverage Month Flag Table

Months below 70 % valid coverage should be treated with caution in downstream
signal computation.  Phoenix summers (June–August) are typically high-coverage
because of low cloud frequency; winter months may suffer from high cirrus.

In [ ]:
if cov_df.empty:
    print('No composites to flag.')
else:
    flagged = cov_df[cov_df['coverage_pct'] < COVERAGE_THRESHOLD].copy()
    flagged['year']  = flagged.index.str[:4].astype(int)
    flagged['month_num'] = flagged.index.str[5:7].astype(int)

    if flagged.empty:
        print(f'All months meet the {COVERAGE_THRESHOLD}% coverage threshold.')
    else:
        print(f'{len(flagged)} months flagged (coverage < {COVERAGE_THRESHOLD}%):')
        display(
            flagged[['coverage_pct', 'valid_px', 'total_px']]
            .rename(columns={
                'coverage_pct': 'Coverage %',
                'valid_px': 'Valid pixels',
                'total_px': 'Total pixels',
            })
            .style.background_gradient(subset=['Coverage %'], cmap='RdYlGn', vmin=0, vmax=100)
            .format({'Coverage %': '{:.1f}', 'Valid pixels': '{:,}', 'Total pixels': '{:,}'})
        )

        # Monthly distribution of low-coverage months
        fig, ax = plt.subplots(figsize=(8, 3))
        flagged['month_num'].value_counts().sort_index().plot.bar(ax=ax, color='#d62728')
        ax.set_xlabel('Calendar month')
        ax.set_ylabel('Count of flagged composites')
        ax.set_title(f'Seasonality of low-coverage months (< {COVERAGE_THRESHOLD}%)')
        ax.set_xticklabels(
            ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
            rotation=0,
        )
        plt.tight_layout()
        plt.show()

In [ ]:
# Summary stats
if not cov_df.empty:
    print('Coverage summary across all months:')
    print(cov_df['coverage_pct'].describe().round(1).to_string())
    print()

    # Check DB registration
    import psycopg2, os
    from dotenv import load_dotenv
    load_dotenv()
    try:
        conn = psycopg2.connect(
            host=os.environ.get('POSTGRES_HOST', 'localhost'),
            port=int(os.environ.get('POSTGRES_PORT', 5432)),
            dbname=os.environ.get('POSTGRES_DB', 'urbangrowth'),
            user=os.environ.get('POSTGRES_USER', 'urbangrowth'),
            password=os.environ.get('POSTGRES_PASSWORD', ''),
        )
        db_df = pd.read_sql(
            "SELECT scene_id, date, cloud_pct, file_path "
            "FROM sentinel_scenes "
            "WHERE scene_id LIKE 'composite_phoenix_%' "
            "ORDER BY date",
            conn,
        )
        conn.close()
        print(f'sentinel_scenes DB rows for Phoenix composites: {len(db_df)}')
        if len(db_df) != len(all_tifs):
            print(f'  WARNING: {len(all_tifs)} COGs on disk but {len(db_df)} rows in DB')
        display(db_df.head(12))
    except Exception as e:
        print(f'DB check skipped: {e}')